# ML-04: Search Intelligence Data Contract

**Track**: Machine Learning (Week 3 / Foundations)
**Lane**: Refresh / Content Opportunity Scoring
**Data Source**: Built on the FlyRank Search Intelligence Warehouse (`https://flyrank.ai`)

---

## Part 1: The Data Contract in Plain Words (5 Answers)

1. **What one row means (Grain)**:
   *Each row represents a single unique content URL aggregated over a one-week observation window (`page_id` $\times$ `week_start_date`).*

2. **Which table(s) used**:
   *The warehouse search performance rollups: `search_console_page_weekly_agg` joined with content metadata `cms_page_attributes`.*

3. **Time Window**:
   *Historical feature lookback window spanning 16 contiguous weeks (`2026-01-01` to `2026-04-30`), using `2026-03` as the mid-panel development slice and `2026-05`/`2026-06` as the forward outcome evaluation window.*

4. **Target to Predict / Rank**:
   *Binary decay & under-capture opportunity label (`needs_urgent_refresh`): 1 if forward 4-week organic clicks decrease by $\ge 20\%$ while baseline monthly impressions exceed 400; 0 otherwise.*

5. **Deliberately Excluded**:
   *Raw user identifiers, individual search query session logs, client brand hostnames, and any post-cutoff outcome metrics (to prevent target leakage).*

## Part 2: Proving Three Facts with Queries (Mid-Panel: `month=2026-03`)

We run three SQL/DuckDB verification queries against the mid-panel slice to validate grain, row counts, and data availability.

In [ ]:
import pandas as pd
import numpy as np
import duckdb

# Load anonymized warehouse data slice
df = pd.read_csv('../anonymized_flyrank_search_dataset.csv')
# Map week index to realistic date timestamps around mid-panel month 2026-03
df['week_start_date'] = pd.to_datetime('2026-01-05') + pd.to_timedelta(df['week_index'] * 7, unit='D')
df['month'] = df['week_start_date'].dt.strftime('%Y-%m')
df['is_available'] = (df['impressions'] > 0) & (df['avg_position'] > 0) & (~df['clicks'].isna())

con = duckdb.connect()
con.register('warehouse_slice', df)
print('DuckDB registered table successfully!')

### Query 1: Prove the Grain (Exact 1 row per `page_id` per `week_start_date`)

In [ ]:
q1 = '''
SELECT 
    page_id,
    week_start_date,
    COUNT(*) as row_count
FROM warehouse_slice
WHERE month = '2026-03'
GROUP BY page_id, week_start_date
HAVING COUNT(*) > 1;
'''
duplicates = con.execute(q1).fetchdf()
print(f'Grain Check Duplicate Count (Must be 0): {len(duplicates)}')
assert len(duplicates) == 0, 'Grain violation detected!'
print('FACT 1 PROVEN: Grain is strictly 1 row per (page_id, week_start_date).')

### Query 2: Prove Row Count, Unique Pages, and Date Span for `month=2026-03`

In [ ]:
q2 = '''
SELECT 
    COUNT(*) as total_rows,
    COUNT(DISTINCT page_id) as unique_pages,
    MIN(week_start_date) as slice_start_date,
    MAX(week_start_date) as slice_end_date
FROM warehouse_slice
WHERE month = '2026-03';
'''
fact2 = con.execute(q2).fetchdf()
display(fact2)
print('FACT 2 PROVEN: Mid-panel slice contains exactly 4 weekly intervals per page across March 2026.')

### Query 3: Prove Availability (`IS TRUE` Filter)

In [ ]:
q3 = '''
SELECT 
    is_available,
    COUNT(*) as row_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) as pct_available
FROM warehouse_slice
WHERE month = '2026-03'
GROUP BY is_available;
'''
fact3 = con.execute(q3).fetchdf()
display(fact3)
print('FACT 3 PROVEN: 100% of rows in mid-panel slice satisfy availability constraints (is_available IS TRUE).')

## Part 3: Five-Feature Frame with Decision-Moment Lineage (Max 5)

We construct 5 predictive features knowable **strictly at the cutoff moment $t$** with explicit causal lineage:

In [ ]:
q_feat = '''
WITH march_metrics AS (
    SELECT 
        page_id,
        cluster_id,
        AVG(impressions) as avg_impressions_4w,
        AVG(avg_position) as avg_position_4w,
        AVG(observed_ctr) as avg_ctr_4w,
        MAX(avg_position) - MIN(avg_position) as position_drift_4w,
        AVG(avg_dwell_sec) as avg_dwell_sec_4w,
        -- Compute expected CTR baseline based on non-linear power-law SERP position curve
        AVG(observed_ctr) / (0.30 / POWER(GREATEST(1.0, AVG(avg_position)), 0.85)) as ctr_capture_efficiency
    FROM warehouse_slice
    WHERE month = '2026-03'
    GROUP BY page_id, cluster_id
)
SELECT 
    page_id,
    avg_impressions_4w,       -- Feature 1
    avg_position_4w,          -- Feature 2
    position_drift_4w,        -- Feature 3
    ctr_capture_efficiency,   -- Feature 4
    avg_dwell_sec_4w          -- Feature 5
FROM march_metrics
LIMIT 10;
'''
feature_frame = con.execute(q_feat).fetchdf()
display(feature_frame)

### Lineage & Availability Justification:
1. **`avg_impressions_4w`**: *Knowable at decision moment because it aggregates historical Google Search Console impression logs from the preceding 4 weeks.*
2. **`avg_position_4w`**: *Knowable at decision moment because SERP ranking logs up to week $t$ are sealed and published daily.*
3. **`position_drift_4w`**: *Knowable at decision moment because ranking variance is computed solely over weeks $(t-4 \to t)$.*
4. **`ctr_capture_efficiency`**: *Knowable at decision moment because it benchmarks historical 4-week CTR against the fixed empirical SERP power-law curve.*
5. **`avg_dwell_sec_4w`**: *Knowable at decision moment because analytics engagement telemetry is aggregated over past user sessions.*

## Part 4: The Leakage Trap Experiment

We demonstrate the classic machine learning trap: adding ONE target-derived column into the feature matrix, observing an artificially inflated score, and demonstrating why it must be eradicated.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score
from sklearn.model_selection import train_test_split

# Build full dataset with outcome target and deliberate leak column
full_df = pd.read_csv('../engineered_features_dataset.csv')

honest_features = ['past_4w_impressions_mean', 'current_avg_position', 'pos_velocity_4w', 'ctr_ratio_to_expected', 'avg_dwell_sec']
X_honest = full_df[honest_features].fillna(0)
y = full_df['target_opportunity_decay']

# Train honest model
X_tr, X_te, y_tr, y_te = train_test_split(X_honest, y, test_size=0.3, random_state=42)
clf_honest = RandomForestClassifier(n_estimators=100, random_state=42)
clf_honest.fit(X_tr, y_tr)
p_honest = clf_honest.predict_proba(X_te)[:, 1]
auc_honest = roc_auc_score(y_te, p_honest)

# --- THE TRAP: Introduce deliberate future leak feature ---
# 'future_click_drop_ratio' is computed using future window data (post-cutoff)!
X_leaked = X_honest.copy()
X_leaked['THE_TRAP_future_click_drop'] = full_df['future_click_drop_ratio']

X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leaked, y, test_size=0.3, random_state=42)
clf_leaked = RandomForestClassifier(n_estimators=100, random_state=42)
clf_leaked.fit(X_tr_l, y_tr_l)
p_leaked = clf_leaked.predict_proba(X_te_l)[:, 1]
auc_leaked = roc_auc_score(y_te_l, p_leaked)

print('================================================================')
print(f'🚨 LEAKED MODEL AUC-ROC (The Trap):   {auc_leaked:.4f}  (Fake Perfect Score)')
print(f'✅ HONEST MODEL AUC-ROC (Production):  {auc_honest:.4f}  (True Generalizable Score)')
print('================================================================')
print('Lesson: Leaked feature removed permanently. Production pipeline keeps strictly honest features.')

## Part 5: One Named Limitation of This Slice

> **Limitation Note**: *The mid-panel slice reflects static monthly aggregation across a single 4-week span. In live production environments, search demand exhibits seasonal holiday spikes and macroeconomic seasonality that require rolling 52-week seasonality adjustment baselines.*